In [1]:
import os
os.environ["NUMBA_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"


In [2]:
import numpy as np
from numba import njit, prange, float32, uint8
from src.constants import *
from time import time, perf_counter


In [3]:
N = len(NEURON_NAMES)
ALPHA_L = 250
td = np.arange(1, ALPHA_L + 1, dtype=np.float32)
alpha = (td / 30) * np.exp((30 - td) / 30)   

cue_wave = np.zeros(TMAX, dtype=np.float32)
go_wave = np.zeros_like(cue_wave)
cue_wave[EPOCHS['sample'][0]:EPOCHS['sample'][1]] = CUE_STRENGTH
go_wave[EPOCHS['response'][0]:EPOCHS['response'][0] + GO_DURATION] = GO_STRENGTH


# --------------------------------------------------------------------
# Hand‑crafted weights -------------------------------------------------
# --------------------------------------------------------------------
new_jh_weights = [
    ("Somat", "ALMprep", 40),
    ("Somat", "MSN1", 220),
    ("MSN1", "SNR1", -90),
    ("SNR1", "VMprep", -10),
    ("VMprep", "ALMprep", 70),
    ("ALMprep", "VMprep", 80),
    ("ALMprep", "MSN2", 320),
    ("MSN2", "SNR2", -50),
    ("SNR2", "VMresp", -100),
    ("PPN", "THALgo", 60),
    ("THALgo", "ALMinter", 55),
    ("ALMinter", "ALMprep", -50),
    ("THALgo", "ALMresp", 30),
    ("ALMresp", "MSN3", 320),
    ("MSN3", "SNR3", -90),
    ("SNR3", "VMresp", -50),
    ("VMresp", "ALMresp", 85),
    ("ALMresp", "VMresp", 90),
]

# --------------------------------------------------------------------
# Build weight matrix -------------------------------------------------
# --------------------------------------------------------------------
N = len(NEURON_NAMES)
W = np.zeros((N, N), dtype=np.float32)
for pre, post, w in new_jh_weights:
    i = NEURON_NAMES.index(pre)
    j = NEURON_NAMES.index(post)
    W[i, j] += w

pass_ids = [NEURON_NAMES.index(x) for x in ["VMresp", "ALMresp", "SNR3"]]
pass_ids = np.array(pass_ids)
print(pass_ids)

[13 12 11]


In [4]:
# CREATING CRITERION
conditions = []
for condition in CRITERIA:
    condition_criteria = []
    for neuron_name, neuron in CRITERIA[condition].items():
        idx = NEURON_NAMES.index(neuron_name)
        baseline = np.ones(TMAX, np.uint8) if neuron_name in TONICALLY_ACTIVE_NEURONS else np.zeros(TMAX, np.uint8)
        start = neuron["interval"][0]
        end = neuron["interval"][1]
        target_status = neuron["io"]
        # print(idx, neuron_name, baseline)
        for i in baseline:
            if target_status == "off":
                baseline[start:end] = 0
            elif target_status == "on":
                baseline[start:end] = 1

        baseline = baseline.reshape(TMAX//BIN_SIZE, BIN_SIZE)
        baseline = np.sum(baseline, axis=1,dtype=np.uint32)
        baseline = (baseline != 0).astype(np.uint8)

        condition_criteria.append((neuron_name, idx, baseline))
    condition_criteria = sorted(condition_criteria, key=lambda tup: tup[1])
    conditions.append(condition_criteria)

crit_Exp, crit_Cont = conditions

crit_indices = np.array([neu[1] for neu in crit_Cont])
crit_Exp = np.vstack([neu[2] for neu in crit_Exp])
crit_Cont = np.vstack([neu[2] for neu in crit_Cont])


In [5]:
# Takes the state at t and updates world to t+1. Returns spikes from step

@njit(parallel=False, fastmath=True, cache=True)
def step_kernel(V, U, Ibuf, t_ptr,
                a, b, vreset, d, k, vr, vt, vpeak, C, E, 
                W, alpha):
    n, L = V.size, alpha.size
    spk  = np.zeros(n, dtype=np.uint8)

    # integrate -------------------------------------------------------
    for i in range(n):
        I = Ibuf[t_ptr, i]
        dV  = (k[i]*(V[i]-vr[i])*(V[i]-vt[i]) - U[i] + I + E[i]) / C[i]
        dU  = a[i]*(b[i]*(V[i]-vr[i]) - U[i])
        V[i] += dV
        U[i] += dU
        if V[i] >= vpeak[i]:
            V[i]  = vreset[i]
            U[i] += d[i]
            spk[i] = 1          # Double check the formula to make sure it aint wonky

    # distribute PSC --------------------------------------------------
    if np.sum(spk) > 0:
        post_I = spk.astype(np.float32) @ W                   # dense GEMV
        t_next = (t_ptr + 1) % L
        for k_shift in range(L):
            Ibuf[(t_next + k_shift) % L, :] += post_I * alpha[k_shift]

    Ibuf[t_ptr,:] = 0.0
    return spk, (t_ptr + 1) % L

In [6]:
@njit(fastmath=True, cache=True)
def score_bin(curr_bin_results, crit_matrix, crit_indices, bin_idx, pass_ids):
    score = 0
    for i in range(len(crit_indices)):
        idx = crit_indices[i]
        if curr_bin_results[idx] == crit_matrix[i, bin_idx]:
            score += 1
        elif (bin_idx * BIN_SIZE > 3500) and (idx in pass_ids):
            score += 1
    return score


In [7]:
# ────────────────────────────────────────────────────────────────────
# 2.  Simulation + scoring
# ────────────────────────────────────────────────────────────────────

@njit(fastmath=True, cache=True)
def simulate(W, 
            a, b, vreset, d, k, vr, vt, vpeak, C, E, 
            alpha, cue_wave, go_wave, 
            crit_Exp, crit_Cont, crit_indices, pass_ids,
            tmax, 
            control, 
            return_full 
            ):

    V = np.full(N, -60.0, np.float32)
    U = np.zeros_like(V, np.float32)
    Ibuf = np.zeros((ALPHA_L, N), dtype=np.float32)
    HIST = np.zeros((N, BIN_SIZE), np.uint8) # 99?
    if return_full:
        temp_full_hist = np.zeros((N, tmax), np.uint8) # 99?

    score = 0
    t_ptr   = 0
    bin = 0

    for t in range(tmax):

        if control == False:
            Ibuf[t_ptr,0] += cue_wave[t]
        Ibuf[t_ptr,7] += go_wave[t]
    
        spk, t_ptr = step_kernel(V, U, Ibuf, t_ptr,
                                 a, b, vreset, d, k, vr, vt, vpeak, C, E, 
                                 W, alpha)

        if return_full:
            temp_full_hist[:,t] = spk 

        cidx = t % BIN_SIZE
        HIST[:,cidx] = spk
        # bit-pack history
        if cidx == (BIN_SIZE - 1):
            curr_bin_results = (np.sum(HIST, axis=1) >= 1).astype(np.uint8)
            crits = crit_Exp if (control == False) else crit_Cont
            score += score_bin(curr_bin_results,crits, crit_indices, bin, pass_ids)
            bin += 1

    return score, (temp_full_hist if return_full else None)


In [8]:
start = perf_counter()
simulate(W, a, b, vreset, d, k, vr, vt, vpeak, C, E,
         alpha, cue_wave, go_wave,
         crit_Exp, crit_Cont, crit_indices, pass_ids,
         5000, False, False)
mid = perf_counter()
# print(mid-start)
# run_batch(W, a, b, vreset, d, k, vr, vt, vpeak, C, E,
#           alpha, cue_wave, go_wave,
#           crit_Exp, crit_Cont, crit_indices, pass_ids,
#           TMAX)
end = perf_counter()
print(f'Total time: {end - start:.3f}s')


Total time: 1.588s


In [24]:
@njit(cache=True)
def run_batch(W, a, b, vreset, d, k, vr, vt, vpeak, C, E,
              alpha, cue_wave, go_wave,
              crit_Exp, crit_Cont, crit_indices, pass_ids,
              tmax, NTRIALS=10):

    total_time = 0.0
    for i in range(NTRIALS):
        s1, _ = simulate(W, a, b, vreset, d, k, vr, vt, vpeak, C, E,
                         alpha, cue_wave, go_wave,
                         crit_Exp, crit_Cont, crit_indices, pass_ids,
                         tmax, False, False)
        s2, _ = simulate(W, a, b, vreset, d, k, vr, vt, vpeak, C, E,
                         alpha, cue_wave, go_wave,
                         crit_Exp, crit_Cont, crit_indices, pass_ids,
                         tmax, True, False)
    return s1, s2

In [30]:
start = perf_counter()
s1,s2=run_batch(W, a, b, vreset, d, k, vr, vt, vpeak, C, E,
          alpha, cue_wave, go_wave,
          crit_Exp, crit_Cont, crit_indices, pass_ids,
          TMAX)
end = perf_counter()
print(f'Total time: {end - start:.3f}s')


Total time: 0.300s


# Runtimes for 10 runs
- No JIT: 6 seconds
- Step Kernel @njit(parallel=False, fastmath=True, cache=True) without prange: 0.592s
- Step Kernel @njit(parallel=False, fastmath=True, cache=True) with prange: 0.555s
- Step Kernel @njit(parallel=True, fastmath=True, cache=True) without prange: 22.615s 
- Step Kernel @njit(parallel=True, fastmath=True, cache=True) with prange: 21.592s
- Step Kernel and Score Bin JIT: 0.569s
- Step Kernel and Score Bin and Simulate JIT: HUNG

In [31]:
print(s1, s2)


470 498


# Config param setup
```
GA_CONFIG={
    "small": {
        "NUM_GENERATIONS" : 5,
        "POP_SIZE" : 50,
        "MUT_RATE" : 0.3,
        "MUT_SIGMA" : 0.5,
        "RANK_DEPTH" : 25,
        "ELITE_SIZE" : 5,
        "CROSSOVER_POINT" : None, # Randomly selecting all genes
        "DNA_BOUNDS" : [0,1000]
    },...
}
```

In [39]:
# Generating random matrices
size = "small"
base_vector = np.ones(len(ACTIVE_SYNAPSES))*GA_CONFIG[size]["DNA_BOUNDS"][1]
for idx, conn in enumerate(ACTIVE_SYNAPSES):
    if conn[0] in INHIBITORY_NEURONS:
        base_vector[idx] *= -1

random_vector = base_vector*[np.random.random(len(base_vector))]
print(random_vector)
# --------------------------------------------------------------------
# Build weight matrix -------------------------------------------------
# --------------------------------------------------------------------
N = len(NEURON_NAMES)
W = np.zeros((N, N), dtype=np.float32)
for pre, post, w in new_jh_weights:
    i = NEURON_NAMES.index(pre)
    j = NEURON_NAMES.index(post)
    W[i, j] += w


[[ 655.74798968  610.81265432  390.86090486  193.86722951 -209.76498759
  -795.67163696 -411.45619848 -204.07396252 -208.24826437 -334.13701304
  -115.82142259 -841.13316999 -885.93143157 -816.48341469 -211.94729205
  -514.31615039 -367.44903907 -334.98695389 -802.24829941  536.17735965
   447.85874155  271.91997159  394.46692916  236.17039661   77.95358643
   999.13295919  361.87523375  125.1044596  -563.50735775 -272.61998123
  -444.03248073  311.65532589  938.34889966  395.03767408  135.67174658
   102.0019442   434.60643837  959.96750926 -182.8631444  -509.26975663
  -175.90620085 -335.10770372 -688.78277333 -541.50694317  227.97835861
   911.98446047 -349.99358248 -235.10668556  788.04420707  433.62734763
   669.82018391   60.31036266  541.35836016]]


In [ ]:
# Add these imports at the top of your notebook
import os
import numba
from time import perf_counter

# Quick Numba test function
def quick_numba_test():
    """Quick test to see immediate Numba effects."""
    print("Quick Numba Test")
    print("=" * 40)
    
    # Test with JIT enabled
    print("\n1. Testing with JIT enabled...")
    os.environ["NUMBA_DISABLE_JIT"] = "0"
    numba.config.DISABLE_JIT = False
    
    start = perf_counter()
    for i in range(5):
        experimental_score, _ = simulate(W, cue_wave, go_wave, tmax=TMAX, control=False)
        control_score, _ = simulate(W, cue_wave, go_wave, tmax=TMAX, control=True)
    end = perf_counter()
    jit_enabled_time = (end - start) / 5
    print(f"JIT Enabled: {jit_enabled_time:.4f}s per run")
    
    # Test with JIT disabled
    print("\n2. Testing with JIT disabled...")
    os.environ["NUMBA_DISABLE_JIT"] = "1"
    numba.config.DISABLE_JIT = True
    
    start = perf_counter()
    for i in range(5):
        experimental_score, _ = simulate(W, cue_wave, go_wave, tmax=TMAX, control=False)
        control_score, _ = simulate(W, cue_wave, go_wave, tmax=TMAX, control=True)
    end = perf_counter()
    jit_disabled_time = (end - start) / 5
    print(f"JIT Disabled: {jit_disabled_time:.4f}s per run")
    
    # Reset to enabled
    os.environ["NUMBA_DISABLE_JIT"] = "0"
    numba.config.DISABLE_JIT = False
    
    print(f"\nSpeedup with JIT: {jit_disabled_time/jit_enabled_time:.2f}x")

# Run the test
quick_numba_test()

In [ ]:

# ────────────────────────────────────────────────────────────────────
# 3.  Example run
# ────────────────────────────────────────────────────────────────────
if __name__ == "__main__":

    t0 = time.time()
    simulate(W, cue_wave, go_wave)
    # score, hist = simulate(W, cue_wave, go_wave)
    # print(f"TOTAL score: {score}")
    print(f"Wall-time: {1000*(time.time()-t0):.1f} ms")

    # unpack if needed
    # bits = np.unpackbits(hist, axis=1)[:, :TMAX]   # shape (N,TMAX)
    # print("Neuron-0 first 40 ms spikes:", bits[0, :40].tolist())